# Shifting the origin

Writing the same parabola in the coordinate $z = x + 10$ changes the intercept and the linear coefficient. It does not change the coefficient of the square, and it does not change any fitted value. The origin of $x$ is a choice of parameterization.


## Expand $1 + 2x + 3x^2$ at $z = x + 10$

Substitute $x = z - 10$:

$$
\begin{aligned}
1 + 2(z - 10) + 3(z - 10)^2
&= 1 + 2z - 20 + 3(z^2 - 20z + 100) \\
&= 281 - 58z + 3z^2.
\end{aligned}
$$

The inputs $x = -2, -1, 0, 1, 2$ become $z = 8, 9, 10, 11, 12$. The coefficient of $z^2$ is still $3$, because a shift does not change a second difference. The intercept $281$ is the prediction at $z = 0$, which is $x = -10$, far from the sample. The linear coefficient $-58$ is the slope of the parabola at $z = 0$, not the slope at the center of the data.

The centered calculation is the reverse substitution. At $z = 10$, which is $x = 0$,

$$
281 - 58\cdot 10 + 3\cdot 100 = 281 - 580 + 300 = 1,
$$

the original intercept. Fitted values are invariant. Coefficients below the top degree are not.


In [1]:
# z = x + 10 rewrites the parabola as 281 - 58z + 3z^2 and keeps every value.
from fractions import Fraction
x = [-2, -1, 0, 1, 2]
z = [xi + 10 for xi in x]
old = [1 + 2 * xi + 3 * xi ** 2 for xi in x]
new = [281 - 58 * zi + 3 * zi ** 2 for zi in z]
print("z", z)
print("values", new)
assert new == old
assert 281 - 58 * 10 + 3 * 100 == 1


z [8, 9, 10, 11, 12]
values [9, 2, 1, 6, 17]


## Fitting in the shifted coordinate

Regress the same responses $10, 0, 1, 8, 16$ on $1$, $z$, and $z^2$. The column space is the same as the column space of $1$, $x$, and $x^2$, because $z$ is an affine function of $x$ and $z^2$ expands into $1$, $z$, and a square. The least-squares predictions must match. The coefficient vector must be the shifted vector of the quadratic solution, plus the shifted residual is the same residual. Since the quadratic solution was $1 + 2x + 3x^2$ and the responses differ from it by $1, -2, 0, 2, -1$, the fitted polynomial in $z$ is exactly $281 - 58z + 3z^2$. The residual vector is unchanged.


In [2]:
# Least squares in z returns 281, -58, 3 and the original residuals.
import numpy as np
x = np.array([-2.0, -1, 0, 1, 2])
z = x + 10
y = np.array([10.0, 0, 1, 8, 16])
Z = np.column_stack([np.ones(5), z, z ** 2])
beta = np.linalg.solve(Z.T @ Z, Z.T @ y)
residuals = y - Z @ beta
print("beta", beta)
print("residuals", residuals)
assert np.allclose(beta, [281, -58, 3])
assert np.allclose(residuals, [1, -2, 0, 2, -1])


beta [281. -58.   3.]
residuals [ 1.00000000e+00 -2.00000000e+00  8.38618064e-12  2.00000000e+00
 -1.00000000e+00]


## One change: center $z$ before squaring

Set $c = z - 10$, which is $x$ again. The columns $1$, $c$, and $c^2$ are the original columns, and the coefficients return to $1, 2, 3$. Centering is the decision to report the intercept at the middle of the sample rather than at $z = 0$. On $z = 8, 9, 10, 11, 12$, the number $281$ is an intercept at a point eleven units away from the nearest observation.


In [3]:
# Centering z recovers the original coefficients 1, 2, 3.
import numpy as np
z = np.array([8.0, 9, 10, 11, 12])
c = z - z.mean()
y = np.array([10.0, 0, 1, 8, 16])
C = np.column_stack([np.ones(5), c, c ** 2])
beta = np.linalg.solve(C.T @ C, C.T @ y)
print("centered beta", beta)
print("mean of z", z.mean())
assert np.allclose(beta, [1, 2, 3])
assert z.mean() == 10


centered beta [1. 2. 3.]
mean of z 10.0


## Pitfall

Comparing $b_1 = 2$ in the $x$ coordinate with $b_1 = -58$ in the $z$ coordinate, and concluding that the association changed sign, compares two different parameters. Both polynomials are the same function. The derivative makes the distinction precise:

$$
\dfrac{d}{dx}(1 + 2x + 3x^2) = 2 + 6x, \qquad
\dfrac{d}{dz}(281 - 58z + 3z^2) = -58 + 6z.
$$

At the matching points $x = 0$ and $z = 10$ both derivatives equal $2$. At $z = 0$ the second derivative expression equals $-58$, which is the slope at $x = -10$, not a property of the sample.


In [4]:
# The two formulas have the same derivative where the coordinates match.
slope_at_center = 2 + 6 * 0
slope_from_shifted = -58 + 6 * 10
print("both slopes", slope_at_center, slope_from_shifted)
assert slope_at_center == slope_from_shifted == 2


both slopes 2 2


## Summary

- In $z = x + 10$ the fitted polynomial is $281 - 58z + 3z^2$. The square's coefficient stays $3$.
- Predictions and residuals are unchanged. The intercept $281$ is the extrapolation to $z = 0$.
- Centering $z$ returns the coefficients $1, 2, 3$ and places the intercept at the sample mean of the predictor.
